# Annotation d'unités factuelles (API OpenAI)

Notebook pas-à-pas pour annoter des récits d'accidents :
- macro-label **A0 / A1 / B / C**
- variables **injury_mentioned**, **hospitalized**, **fatal**
- cache **JSONL** local pour reprise sans tout refaire
- **prompt caching OpenAI** (`prompt_cache_key` + `cached_tokens` dans les logs)
- snapshots XLSX périodiques

**Prérequis** : `OPENAI_API_KEY` dans `text/.env` (voir `.env.example`).
Placez votre CSV dans `annotation/data/`.


In [1]:
import os
import sys
from pathlib import Path


def _notebook_find_text_root(start: Path) -> Path:
    here = start.resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "safer_core" / "paths.py").is_file():
            return candidate
        nested = candidate / "text"
        if (nested / "safer_core" / "paths.py").is_file():
            return nested
    raise FileNotFoundError(
        "Racine text/ introuvable (safer_core/paths.py). "
        "Ouvrez Jupyter depuis le dossier text/ ou SAFER/."
    )


TEXT_ROOT = _notebook_find_text_root(Path.cwd())
if str(TEXT_ROOT) not in sys.path:
    sys.path.insert(0, str(TEXT_ROOT))
os.chdir(TEXT_ROOT)


In [7]:
# --- Paramètres ---
from pathlib import Path

INPUT_CSV = "btp_sentence_accidents.csv"          # relatif à annotation/data/
OUTPUT_BASENAME = "btp_v10_gpt5_mini"
OPENAI_MODEL = "gpt-5-mini"
REASONING_EFFORT = "medium"
PROMPT_VERSION = "v10_macro_labels_independent_outcomes"
PROMPT_CACHE_KEY = None               # None → safer-annotation:{PROMPT_VERSION}
USE_PROMPT_CACHE_KEY = True           # prompt caching OpenAI (préfixe system identique)

N_ACCIDENTS = 10                      # int ou "all"
UNITS_PER_ACCIDENT = "all"            # int ou "all"
ACCIDENT_SAMPLE_SEED = 42

TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 4000          # budget raisonnement (medium) + JSON
SAVE_EVERY = 50
MAX_RETRIES = 3
RETRY_BASE_SLEEP_SEC = 1.5
RATE_LIMIT_SLEEP_SEC = 30.0
MIN_DELAY_BETWEEN_CALLS_SEC = 0.5

SKIP_CACHE = False                    # True = ignorer le cache JSONL
DRY_RUN = False                       # True = prévisualiser sans appeler l'API
SUMMARY_COL = "accident_summary"      # ou summary_accident

RUN_ID = None                          # ex. reprendre une run existante

ANNOTATION_ROOT = TEXT_ROOT / "annotation"
print("ANNOTATION_ROOT =", ANNOTATION_ROOT)
print("Modèle :", OPENAI_MODEL, "| reasoning :", REASONING_EFFORT)


ANNOTATION_ROOT = C:\Users\aho\Documents\analysis factor project\SAFER\text\annotation
Modèle : gpt-5-mini | reasoning : medium


In [8]:
# --- Étape 1 : chargement CSV ---
import pandas as pd
from annotation.config import AnnotationConfig

cfg = AnnotationConfig(
    input_csv=INPUT_CSV,
    output_basename=OUTPUT_BASENAME,
    openai_model=OPENAI_MODEL,
    prompt_version=PROMPT_VERSION,
    prompt_cache_key=PROMPT_CACHE_KEY,
    use_prompt_cache_key=USE_PROMPT_CACHE_KEY,
    n_accidents=N_ACCIDENTS,
    units_per_accident=UNITS_PER_ACCIDENT,
    accident_sample_seed=ACCIDENT_SAMPLE_SEED,
    temperature=TEMPERATURE,
    reasoning_effort=REASONING_EFFORT,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    save_every=SAVE_EVERY,
    max_retries=MAX_RETRIES,
    retry_base_sleep_sec=RETRY_BASE_SLEEP_SEC,
    rate_limit_sleep_sec=RATE_LIMIT_SLEEP_SEC,
    min_delay_between_calls_sec=MIN_DELAY_BETWEEN_CALLS_SEC,
    skip_cache=SKIP_CACHE,
    dry_run=DRY_RUN,
    summary_col=SUMMARY_COL,
    annotation_root=ANNOTATION_ROOT,
    run_id=RUN_ID,
)

input_path = cfg.resolved_input_path
print(f"Chargement : {input_path}")
df_raw = pd.read_csv(input_path)
print(f"Lignes totales : {len(df_raw):,}")
if "fact_id" not in df_raw.columns:
    print("⚠ fact_id absent — les clés de cache utiliseront ROWIDX_*")
cfg.validate_input_columns(list(df_raw.columns))
df_raw.head(3)


Chargement : C:\Users\aho\Documents\analysis factor project\SAFER\text\annotation\data\btp_sentence_accidents.csv
Lignes totales : 42,309


,accident_id,fact_id,sentence,Unnamed: 0,division,equipment_involved,company_code,accident_summary,word_count
0,EB16DB93EC8D4705C1258D7A002EF784,1,L'accident de travail a eu lieu sur un chantie...,20,43,510328 - Autre type d'échafaudage,4391B - Travaux de couverture par éléments,L'accident de travail a eu lieu sur un chantie...,32
1,EB16DB93EC8D4705C1258D7A002EF784,2,Un échafaudage de pied est positionné sur un m...,20,43,510328 - Autre type d'échafaudage,4391B - Travaux de couverture par éléments,L'accident de travail a eu lieu sur un chantie...,10
2,EB16DB93EC8D4705C1258D7A002EF784,3,L'échafaudage nest pas conforme : il est situ...,20,43,510328 - Autre type d'échafaudage,4391B - Travaux de couverture par éléments,L'accident de travail a eu lieu sur un chantie...,60


In [9]:
# --- Étape 2 : sous-échantillonnage accidents / unités ---
from annotation.runner import prepare_annotation_frame
from annotation.sampling import sampling_stats

df_work = prepare_annotation_frame(cfg, df_raw)
stats = sampling_stats(df_work)
print("Sous-ensemble :")
for k, v in stats.items():
    print(f"  {k}: {v:,}" if isinstance(v, int) else f"  {k}: {v}")
if DRY_RUN:
    print("DRY_RUN=True → aucun appel API ne sera effectué.")
else:
    print(f"Appels API estimés : {stats['estimated_api_calls']:,}")
df_work.head(5)


Sous-ensemble :
  n_rows: 54
  n_accidents: 10
  estimated_api_calls: 54
Appels API estimés : 54


,accident_id,fact_id,sentence,Unnamed: 0,division,equipment_involved,company_code,accident_summary,word_count
0,09F934A4086DF42FC125719D00517D2B,39399,"La victime - plombier chauffagiste, âgé de 19 ...",21065,45,050900 - Moyen de transport sur route non précisé,452BD - Travaux de gros oeuvre et organisation...,"La victime - plombier chauffagiste, âgé de 19 ...",35
1,F524C3F0ACC19F33C12582E90042296F,7949,Il s'agit du premier jour de travail sur le ch...,3061,42,410902 - Rail et traverse,4211Z - Construction de routes et autoroutes,Il s'agit du premier jour de travail sur le ch...,27
2,F524C3F0ACC19F33C12582E90042296F,7950,Les rails en bord de bâtiment ont été découpés.,3061,42,410902 - Rail et traverse,4211Z - Construction de routes et autoroutes,Il s'agit du premier jour de travail sur le ch...,9
3,F524C3F0ACC19F33C12582E90042296F,7951,Le premier rail est noyé dans le béton et liai...,3061,42,410902 - Rail et traverse,4211Z - Construction de routes et autoroutes,Il s'agit du premier jour de travail sur le ch...,17
4,F524C3F0ACC19F33C12582E90042296F,7952,Il est arraché à son support au moyen d'un lev...,3061,42,410902 - Rail et traverse,4211Z - Construction de routes et autoroutes,Il s'agit du premier jour de travail sur le ch...,24


In [10]:
# --- Étape 3 : chemins de sortie ---
print("run_id =", cfg.run_id)
print("outputs_dir =", cfg.outputs_dir)
cfg.outputs_dir.mkdir(parents=True, exist_ok=True)


run_id = btp_v10_gpt5_mini__gpt-5-mini__v10_macro_labels_independent_outcomes__20260711T143523Z
outputs_dir = C:\Users\aho\Documents\analysis factor project\SAFER\text\annotation\outputs\btp_v10_gpt5_mini__gpt-5-mini__v10_macro_labels_independent_outcomes__20260711T143523Z


In [11]:
# --- Étape 4 : inspection du cache ---
from annotation.cache import get_output_paths, load_cache, make_cache_key

jsonl_path, snapshot_path, annotated_path, summary_path, accident_path = get_output_paths(
    cfg.outputs_dir,
    model_id=cfg.openai_model,
    prompt_version=cfg.prompt_version,
)
cache = {} if cfg.skip_cache else load_cache(jsonl_path)
n_hits = sum(
    1
    for idx, row in df_work.iterrows()
    if make_cache_key(row, row_idx=idx) in cache
)
print(f"Cache JSONL : {jsonl_path}")
print(f"Entrées en cache : {len(cache):,}")
print(f"Hits attendus sur ce sous-ensemble : {n_hits:,} / {len(df_work):,}")
print(f"Nouveaux appels estimés : {len(df_work) - n_hits:,}")


Cache JSONL : C:\Users\aho\Documents\analysis factor project\SAFER\text\annotation\outputs\btp_v10_gpt5_mini__gpt-5-mini__v10_macro_labels_independent_outcomes__20260711T143523Z\gpt-5-mini__v10_macro_labels_independent_outcomes.jsonl
Entrées en cache : 0
Hits attendus sur ce sous-ensemble : 0 / 54
Nouveaux appels estimés : 54


In [12]:
# --- Étape 5 : annotation (tqdm) ---
from annotation.runner import classify_dataframe_with_cache

df_pred, meta = classify_dataframe_with_cache(df_work, cfg, show_errors=True)
print("Meta :", meta)
df_pred.head(3)


Cache JSONL : 0 entrées depuis C:\Users\aho\Documents\analysis factor project\SAFER\text\annotation\outputs\btp_v10_gpt5_mini__gpt-5-mini__v10_macro_labels_independent_outcomes__20260711T143523Z\gpt-5-mini__v10_macro_labels_independent_outcomes.jsonl
Prompt caching OpenAI : prompt_cache_key='safer-annotation:v10_macro_labels_independent_outcomes'


Annotation OpenAI [gpt-5-mini]:   0%|          | 0/54 [00:00<?, ?ligne/s]

Snapshot sauvegardé : C:\Users\aho\Documents\analysis factor project\SAFER\text\annotation\outputs\btp_v10_gpt5_mini__gpt-5-mini__v10_macro_labels_independent_outcomes__20260711T143523Z\gpt-5-mini__v10_macro_labels_independent_outcomes__snapshot.xlsx
Sauvegarde finale : C:\Users\aho\Documents\analysis factor project\SAFER\text\annotation\outputs\btp_v10_gpt5_mini__gpt-5-mini__v10_macro_labels_independent_outcomes__20260711T143523Z\gpt-5-mini__v10_macro_labels_independent_outcomes__annotated.xlsx
Meta : {'jsonl_path': 'C:\\Users\\aho\\Documents\\analysis factor project\\SAFER\\text\\annotation\\outputs\\btp_v10_gpt5_mini__gpt-5-mini__v10_macro_labels_independent_outcomes__20260711T143523Z\\gpt-5-mini__v10_macro_labels_independent_outcomes.jsonl', 'snapshot_xlsx_path': 'C:\\Users\\aho\\Documents\\analysis factor project\\SAFER\\text\\annotation\\outputs\\btp_v10_gpt5_mini__gpt-5-mini__v10_macro_labels_independent_outcomes__20260711T143523Z\\gpt-5-mini__v10_macro_labels_independent_outcom

,accident_id,fact_id,sentence,accident_summary,pred_label,pred_injury_mentioned,pred_hospitalized,pred_fatal,pred_confidence,pred_justification,...,pred_source,Unnamed: 0,division,equipment_involved,company_code,word_count,usage_tokens,prompt_tokens,completion_tokens,cached_tokens
0,09F934A4086DF42FC125719D00517D2B,39399,"La victime - plombier chauffagiste, âgé de 19 ...","La victime - plombier chauffagiste, âgé de 19 ...",C,NOT_MENTIONED,NOT_MENTIONED,YES,0.95,Contexte utilisé: non\nIndice principal: « acc...,...,new,21065,45,050900 - Moyen de transport sur route non précisé,452BD - Travaux de gros oeuvre et organisation...,35,2606,2071,535,0
1,F524C3F0ACC19F33C12582E90042296F,7949,Il s'agit du premier jour de travail sur le ch...,Il s'agit du premier jour de travail sur le ch...,A0,NOT_MENTIONED,NOT_MENTIONED,NOT_MENTIONED,0.95,Contexte utilisé: non\nIndice principal: « pre...,...,new,3061,42,410902 - Rail et traverse,4211Z - Construction de routes et autoroutes,27,2597,2052,545,1792
2,F524C3F0ACC19F33C12582E90042296F,7950,Les rails en bord de bâtiment ont été découpés.,Il s'agit du premier jour de travail sur le ch...,A0,NOT_MENTIONED,NOT_MENTIONED,NOT_MENTIONED,0.95,Contexte utilisé: non. Indice principal: « Les...,...,new,3061,42,410902 - Rail et traverse,4211Z - Construction de routes et autoroutes,9,2475,2031,444,1792


In [13]:
# --- Étape 6 : résumé des prédictions ---
from annotation.aggregate import summarize_predictions

summary_df = summarize_predictions(df_pred)
print(summary_df.to_string(index=False))
print(f"pred_ok : {df_pred['pred_ok'].sum():,} / {len(df_pred):,}")
if meta.get("total_tokens"):
    print(f"Tokens totaux (run) : {meta['total_tokens']:,}")
if meta.get("total_prompt_tokens"):
    print(f"Tokens prompt : {meta['total_prompt_tokens']:,}")
if meta.get("total_cached_tokens"):
    print(f"Tokens prompt cachés (OpenAI) : {meta['total_cached_tokens']:,}")
    if meta.get("prompt_cache_hit_rate") is not None:
        print(f"Taux cached/prompt : {meta['prompt_cache_hit_rate']:.1%}")


           level         value  count
           label            A0     26
           label             C     10
           label            A1     10
           label             B      8
injury_mentioned NOT_MENTIONED     50
injury_mentioned           YES      4
    hospitalized NOT_MENTIONED     52
    hospitalized           YES      2
           fatal NOT_MENTIONED     46
           fatal           YES      8
    context_used         False     54
   prediction_ok          True     54
pred_ok : 54 / 54
Tokens totaux (run) : 138,290
Tokens prompt : 110,642
Tokens prompt cachés (OpenAI) : 93,184
Taux cached/prompt : 84.2%


In [14]:
# --- Étape 7 : agrégation au niveau accident ---
from annotation.aggregate import aggregate_outcomes_by_accident
from annotation.export_io import save_annotation_table

accident_df = aggregate_outcomes_by_accident(df_pred)
save_annotation_table(accident_df, meta["accident_xlsx_path"])
print("Sauvegardé :", meta["accident_xlsx_path"])
accident_df.head(10)


Sauvegardé : C:\Users\aho\Documents\analysis factor project\SAFER\text\annotation\outputs\btp_v10_gpt5_mini__gpt-5-mini__v10_macro_labels_independent_outcomes__20260711T143523Z\gpt-5-mini__v10_macro_labels_independent_outcomes__accident_outcomes.xlsx


,accident_id,accident_summary,n_factual_units,n_valid_predictions,n_context_used_units,accident_any_context_used,accident_injury_mentioned,accident_hospitalized,accident_fatal,injury_annotation_conflict,hospitalization_annotation_conflict,fatal_annotation_conflict
0,09F934A4086DF42FC125719D00517D2B,"La victime - plombier chauffagiste, âgé de 19 ...",1,1,0,False,NOT_MENTIONED,NOT_MENTIONED,YES,False,False,False
1,0C12E4E4C406E387C1258A44004FAC57,Un peintre de 56 ans était affecté sur le chan...,7,7,0,False,YES,NOT_MENTIONED,YES,False,False,False
2,4F4F25B54CFC60ECC125829700497CEB,Un coffreur bancheur de 34 ans était au rez-de...,4,4,0,False,NOT_MENTIONED,NOT_MENTIONED,YES,False,False,False
3,8BB85D454F83E994C1257918004561F3,Un couvreur en charpente métallique de 53 ans ...,7,7,0,False,NOT_MENTIONED,NOT_MENTIONED,NOT_MENTIONED,False,False,False
4,BF2885BCA73CD978C125719D00566C99,"La victime, chef d'équipe de chantier âgé de 5...",8,8,0,False,YES,YES,NOT_MENTIONED,False,False,False
5,E5CF559EE93F9180C125719D005509CD,"L'ouvrier, 49 ans, maçon, se trouvait au-desso...",3,3,0,False,YES,NOT_MENTIONED,YES,False,False,False
6,EFE5458B86ED92DAC125719D00565C8F,"Dans un angle du bâtiment, au niveau du rez-de...",7,7,0,False,NOT_MENTIONED,YES,YES,False,False,False
7,F4E41892B1A6E94DC125719D0052426F,"La victime - monteur de réseau, âgé de 53 ans ...",6,6,0,False,NOT_MENTIONED,NOT_MENTIONED,YES,False,False,False
8,F524C3F0ACC19F33C12582E90042296F,Il s'agit du premier jour de travail sur le ch...,6,6,0,False,YES,NOT_MENTIONED,YES,False,False,False
9,FF1305981F7CFC47C125719D00562C72,"La victime - 46 ans, chef de chantier - se dép...",5,5,0,False,NOT_MENTIONED,NOT_MENTIONED,YES,False,False,False
